## Setting up to send emails from your SMTP server

In [100]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
from agents import set_default_openai_client, set_tracing_disabled
from openai import AsyncOpenAI



load_dotenv(override=True)

True

In [79]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")


if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set")
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

Email address is set
SMTP server is set
App password is set
Email is set up and we will try using it


In [80]:
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [81]:
send_email("Testing testing 123", "Fingers crossed..", "<html><body><h1>Subject : Testing testing 123</h1><strong>Fingers</strong> crossed..</body></html>")

In [82]:
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

## Orchestrating Models

#### 1. Orchestrating by Code
#### 2. Orchestrating by LLM's
####       a) via Tools
####       b) via Handoffs

In [ ]:
intro = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.

"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [ ]:
#default setup openrouter in place of openAi

client = AsyncOpenAI(
    #api_key=,
    base_url="https://openrouter.ai/api/v1",
)

set_default_openai_client(client)

In [85]:
MODEL_NAME = "dots-3-note-preview:free"

sales_agent1 = Agent(name="Professional Email", instructions= instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Friendly Email", instructions= instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Short Email", instructions=instructions3, model=MODEL_NAME)

In [ ]:
# result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
# async for event in result.stream_events():
#     if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
#         print(event.data.delta, end="", flush=True)

## Practise :  Convert the email in HTML format and then send directly through email

In [ ]:

# email_html = """
# take athe following as a HTML document. return only valid HTML with no explanation also remove '''html if written
# """

# response = await Runner.run(sales_agent1, "write an 200 word email in initial Hi [contacts's Name ] = only should be written Dear and at the end write best regards, Sumit Sharma at new line add my number +91 8289051950 not eelse only two labels required")
# response.to_input_list()

# next_input = response.to_input_list() + [{"role": "user", "content": email_html}]

# response = await Runner.run(sales_agent2, next_input)
# print(response.final_output)
# send_email("A less painful way to tackle your next audit", "Fingers crossed..", response.final_output)

In [86]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

[non-fatal] Tracing client error 401. Response data is redacted.




Of course. Here is a cold sales email written in the requested style.

***

**Subject:** Streamlining your SOC2 audit preparation

**Body:**

Hi [Recipient Name],

I'm reaching out from ComplAI, where we help companies like [Recipient's Company Name] prepare for and pass their SOC 2 audits with significantly less stress and overhead.

We understand that navigating the complexities of SOC2 compliance is a major drain on your team's time and resources. Our AI-powered SaaS platform automates the evidence collection, policy mapping, and control monitoring processes, transforming a months-long, manual effort into a streamlined, manageable workflow.

The goal is simple: to give you back your time and ensure audit success, so you can focus on scaling your business with confidence.

Would you be open to a brief 15-minute call next week to see how our platform could work for you?

Best regards,

[Your Name]
Sales Executive, ComplAI
[Your Phone Number]
[Link to ComplAI Website]

---
**P.S.** M

[non-fatal] Tracing client error 401. Response data is redacted.


In [87]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales_picker", instructions=decision, model=MODEL_NAME)

In [88]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.


Best sales email:


**Subject:** Taming the SOC2 Audit Beast: A Demo for [Company Name]

**Body:**

Dear [Contact Name],

I hope this email finds you well. My name is [Your Name], and I lead business development at ComplAI.

I'm reaching out because I noticed [Company Name] is likely in a growth phase, and with that comes the critical need to establish trust with enterprise clients and investors. A SOC2 Type II report is no longer a "nice-to-have" but a fundamental requirement for that trust.

The challenge, as I understand it, is that the traditional path to SOC2 compliance is incredibly time-consuming and resource-intensive. It often means months of manual evidence collection, disjointed spreadsheets, and the constant anxiety of whether you'll be audit-ready when the time comes.

This is exactly the problem we built ComplAI to solve.

ComplAI is an AI-powered platform that automates the evidence collection and control monitoring process for SOC2. Instead of a chaotic, manual effort, 

[non-fatal] Tracing client error 401. Response data is redacted.


# Now converting the same email funtion  into tool.

In [89]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

### This has automatically been converted into a tool, with the boilerplate json created

In [73]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [90]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required")

sales_sender = Agent(name="Sales Sender", instructions=decision, model=MODEL_NAME, tools=[send_email_tool], model_settings=require_tool)

In [ ]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.


Final response:


I've selected and sent the witty, engaging, and humorous cold sales email. This version stands out because it creates a memorable human connection through relatable humor while still being professional and clearly communicating the value proposition. The creative approach, including the playful P.S. with "Surprise me," makes it more likely to cut through the noise and generate a positive response compared to more standard sales emails.


[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.



#### 2. Orchestrating by LLM's
####       2a) via Tools


In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

#Example commented below

#tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)

FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x116ae7e00>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [93]:
tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

tools

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x116aeaba0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None),
 FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema

In [94]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=MODEL_NAME)

In [97]:
task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.


[non-fatal] Tracing client error 401. Response data is redacted.


#### 2. Orchestrating by LLM's

Agents-as-tools are similar and Handoffs :

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

A -> B -> A

With handoffs, control passes across

A -> B


####       2b) via Handoffs

In [98]:
instructions = """
You are a Sales Manager at ComplAI. You get your sales team to draft emails, then send them all to a sales picker.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Handoff to the sales sender to choose and send the best email.
"""

tools = [tool1, tool2, tool3]
handoffs = [sales_sender]

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, handoffs=handoffs, model=MODEL_NAME)


In [99]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.


[non-fatal] Tracing client error 401. Response data is redacted.
